9/21

###Spatiotemporal 4D Modeling of Glioblastoma Using Microenviornment-Aware Neural Operators Physics-Regularized Dynamics

##Phase 1 - Multimodal Data Pipeline and Microenvironmental Mapping

**Lucia Nanda**

------------------------

Install TCIA Tools & Core Libraries

In [1]:
# 1. GPU Check
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Install TCIA Python Client and Imaging Stack
!pip install -q tciaclient torchio monai dipy nibabel antspyx

Tue Sep 22 01:06:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Establish Working Directory Hierarchy

In [2]:
import os
from pathlib import Path

# Google Drive storage (For persistent model weights, logs, small outputs)
DRIVE_DIR = Path('/content/drive/MyDrive/GBM_4D_Modeling')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Local Colab disk (For downloading active subjects temporarily)
LOCAL_DATA_DIR = Path('/content/temp_gbm_data')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Drive path: {DRIVE_DIR}")
print(f"Local temp path: {LOCAL_DATA_DIR}")

Drive path: /content/drive/MyDrive/GBM_4D_Modeling
Local temp path: /content/temp_gbm_data


TCIA API Connectivity & Subject Selector

In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

class ReliableTCIAClient:
    def __init__(self, collection="UPENN-GBM"):
        self.base_url = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
        self.collection = collection
        self.session = requests.Session()

        # Configure robust retries for TCIA endpoint stability
        retries = Retry(
            total=5,
            backoff_factor=1,
            status_forcelist=[500, 502, 503, 504, 104],
            raise_on_status=False
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retries))
        self.session.headers.update({"User-Agent": "Mozilla/5.0"})

    def get_patient_ids(self):
        """Fetches all subject IDs directly via TCIA REST API."""
        url = f"{self.base_url}/getPatient?Collection={self.collection}&format=json"
        response = self.session.get(url, timeout=30)

        if response.status_code == 200:
            data = response.json()
            if isinstance(data, list):
                # Using 'PatientId' (lowercase 'd') to match TCIA API JSON payload key
                patient_ids = sorted(list(set([item.get("PatientId") or item.get("PatientID") for item in data if "PatientId" in item or "PatientID" in item])))
                print(f"Successfully connected! Found {len(patient_ids)} subjects in {self.collection}.")
                return patient_ids
            else:
                print("API Warning: Received unexpected non-list payload:", data)
                return []
        else:
            raise Exception(f"Failed to fetch patients. Status code: {response.status_code}")

    def get_patient_series(self, patient_id):
        """Gets series metadata for a single patient."""
        url = f"{self.base_url}/getSeries?Collection={self.collection}&PatientID={patient_id}&format=json"
        response = self.session.get(url, timeout=30)
        if response.status_code == 200:
            return response.json()
        return []

# Initialize client
client = ReliableTCIAClient(collection="UPENN-GBM")
subject_ids = client.get_patient_ids()

# Inspect first 5 subjects
if subject_ids:
    print("Sample Subject IDs:", subject_ids[:5])

Successfully connected! Found 630 subjects in UPENN-GBM.
Sample Subject IDs: ['UPENN-GBM-00001', 'UPENN-GBM-00002', 'UPENN-GBM-00003', 'UPENN-GBM-00004', 'UPENN-GBM-00005']


Access patient data and test on `UPENN-GBM-00001`

In [4]:
import zipfile

def download_and_extract_subject(patient_id, client, output_dir):
    subject_dir = Path(output_dir) / patient_id
    subject_dir.mkdir(parents=True, exist_ok=True)

    series_list = client.get_patient_series(patient_id)
    print(f"\n--- Downloading {len(series_list)} Series for Subject: {patient_id} ---")

    for s in series_list:
        series_uid = s.get("SeriesInstanceUID")
        modality = s.get("Modality", "UNK")
        description = s.get("SeriesDescription", "NoDescription")

        image_url = f"{client.base_url}/getImage?SeriesInstanceUID={series_uid}"
        response = client.session.get(image_url, stream=True, timeout=60)

        if response.status_code == 200:
            zip_path = subject_dir / f"{modality}_{series_uid[:8]}.zip"
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            extract_folder = subject_dir / f"{modality}_{description.replace(' ', '_')}"
            extract_folder.mkdir(exist_ok=True)
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_folder)
                zip_path.unlink()
                print(f" -> Retrieved [{modality}]: {description}")
            except zipfile.BadZipFile:
                print(f" -> Skipping corrupt series [{modality}]: {description}")
        else:
            print(f" -> Failed to fetch {modality} series: Status {response.status_code}")

    return subject_dir

# Fetch UPENN-GBM-00001
sample_patient = subject_ids[0]
patient_folder = download_and_extract_subject(sample_patient, client, LOCAL_DATA_DIR)


--- Downloading 6 Series for Subject: UPENN-GBM-00001 ---
 -> Retrieved [MR]: t1 axial: Processed_CaPTk
 -> Retrieved [MR]: ep2d_perf 12 CC BOLUS
 -> Retrieved [MR]: t1 axial stealth-post : Processed_CaPTk
 -> Retrieved [MR]: t2_Flair_axial: Processed_CaPTk
 -> Retrieved [MR]: ep2d_diff_MDDW_IPAT
 -> Retrieved [MR]: T2 SAG SPACE: Processed_CaPTk


Todo:
```Google Drive/
└── UPenn_GBM_Processed/
    ├── UPENN-GBM-00001/
    │   ├── D_tract.pt      # Spatial tensor matrix (float16)
    │   ├── FA.pt           # Scalar field (float16)
    │   ├── M_barrier.pt    # Binary boundary mask
    │   ├── Q_ECM.pt        # ECM density field
    │   └── H_N_profile.pt  # Hypoxia & Nutrient continuous fields
    ├── UPENN-GBM-00002/
    │   └── ...
    └── cohort_manifest.csv # Log tracking included vs. excluded patients (google sheet)